# Random Forest

This notebook trains and evaluates the Random Forest using the best hyperparameters found in `main_M.ipynb` by 5-fold cross-validation.

**Project location assumption:** save this notebook as `imputer/randomforest.ipynb`, alongside `main_M.ipynb`. It reads `../data/data.csv` and imports `utils.py` from the project root.

In [ ]:
import pandas as pd
import numpy as np
from pandas.api.types import is_numeric_dtype

In [ ]:
df = pd.read_csv("../data/data.csv")

In [ ]:
def display_nan_stats(df):
    nan_stats = pd.DataFrame({
        "dtype": df.dtypes,
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2),
        "unique_count": df.nunique(),
        "values": [
            "" if is_numeric_dtype(df[col])
            else df[col].dropna().unique().tolist()
            for col in df.columns
        ]
    })

    nan_stats = nan_stats.sort_values(
        "missing_percent",
        ascending=False
    )

    with pd.option_context(
        "display.max_rows", None,
        "display.max_colwidth", None
    ):
        display(nan_stats)

    return nan_stats

display_nan_stats(df)

In [ ]:
def fill_missing_false_for_true_only(df):
    df = df.copy()
    processed_columns = []

    for col in df.columns:
        values = set(df[col].dropna().unique())

        if values == {True}:
            df[col] = df[col].fillna(False)
            processed_columns.append(col)

    print(f"Processed {len(processed_columns)} columns:")
    print(processed_columns)

    return df

df = fill_missing_false_for_true_only(df)
nan_stats = display_nan_stats(df)

In [ ]:
start_features = ['aus_boden_kunststoff_janein',
                  'aus_boden_laminat_janein',
                  'aus_boden_linoleum_janein',
                  'aus_boden_marmor_janein',
                  'aus_balkon_janein',
                  'aus_boden_estrich_janein', 
                  'aus_terrasse_janein',
                  'aus_boden_fliesen_janein',
                  'aus_boden_terrakotta_janein',
                  'aus_boden_parkett_janein',
                  'aus_heiz_fernwaerme_janein',
                  'aus_heiz_fussboden_janein',
                  'aus_kueche_einbau_janein',
                  'aus_kueche_offen_janein',
                  'oeig_neubau_janein',
                  'objekttyp_fein',
                  'aus_boden_teppich_janein',
                  'flaeche', # new features
                  'oeig_keller_janein',
                  'baujahr',
                  'etage',
                  'anz_zimmer',
                  'aus_klassen5',
                  'aadr_gewerblich_janein',
                  'aus_alarmanlage_janein',
                  'aus_abstellraum_janein',
                  'anzart_indexmiete_janein',
                  'anzart_nachmieter_janein',
                  'anzart_genossenschaft_janein',
                  'aus_bad_dusche_janein',
                  'aus_bad_mit_fenster_janein',
                  'aus_bad_gaestewc_janein',
                  'aus_barrierefrei_janein',
                  'aus_bad_wanne_janein',
                  'energie_niedrig_janein',
                  'aus_pool_janein',
                  'aus_sauna_janein',
                  'aus_parken_tiefgarage_janein',
                  'aus_hauswirtschaftsraum_janein',
                  'aus_garten_janein',
                  'aus_kamin_janein',
                  'aus_lastenaufzug_janein',
                  'aus_lift_janein',
                  'aus_parken_garage_janein',
                  'aus_parken_janein',
                  'aus_parken_ladestation_janein',
                  'oeig_loggia_janein',
                  'oeig_galerie_janein',
                  'zust_gepflegt_janein',
                  'zust_projektiert_janein',
                  'zust_sanierungsbedarf_janein',
                 ]

df = df[start_features + ['kosten_je_flaeche']].copy()

In [ ]:
df = pd.get_dummies(
    df,
    columns=["objekttyp_fein", 'aus_klassen5'],
    dtype=int
) # one-hot encoding 
df = df.dropna(subset=df.select_dtypes(include="object").columns) # numeric NaN are imputed with EM later

In [ ]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype("boolean")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from utils import train_model, evaluate_model

In [ ]:
target = "kosten_je_flaeche"

X = df.drop(columns=[target])
y = df[target]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
class EMImputer:
    """
    Impute missing values with the EM algorithm under a multivariate
    Gaussian model.

    fit() estimates the mean and covariance from data with NaNs.
    transform() replaces each NaN with its conditional expectation
    given the observed values in the same row.

    Columns are standardized internally; a small ridge term keeps the
    covariance invertible (constant or collinear columns, e.g. one-hot).
    """

    def __init__(self, max_iter=100, tol=1e-4, ridge=1e-6):
        self.max_iter = max_iter
        self.tol = tol
        self.ridge = ridge

    def fit(self, X):
        Z, mask = self._standardize(X, fit=True)
        n, d = Z.shape

        mu = np.zeros(d)
        sigma = np.cov(np.where(mask, 0.0, Z), rowvar=False)

        for it in range(self.max_iter):
            Z_hat, C = self._e_step(Z, mask, mu, sigma)

            mu_new = Z_hat.mean(axis=0)
            diff = Z_hat - mu_new
            sigma_new = (diff.T @ diff + C) / n

            change = max(
                np.abs(mu_new - mu).max(),
                np.abs(sigma_new - sigma).max()
            )
            mu, sigma = mu_new, sigma_new

            if change < self.tol:
                break

        self.mean_ = mu
        self.covariance_ = sigma
        self.n_iter_ = it + 1

        return self

    def transform(self, X):
        Z, mask = self._standardize(X, fit=False)
        Z_hat, _ = self._e_step(Z, mask, self.mean_, self.covariance_)
        X_hat = Z_hat * self.scale_ + self.center_

        if isinstance(X, pd.DataFrame):
            return pd.DataFrame(X_hat, index=X.index, columns=X.columns)

        return X_hat

    def _standardize(self, X, fit):
        # via pandas so nullable "boolean" columns turn <NA> into NaN
        X = pd.DataFrame(X).astype(float).to_numpy()

        if fit:
            self.center_ = np.nanmean(X, axis=0)
            scale = np.nanstd(X, axis=0)
            self.scale_ = np.where(scale > 0, scale, 1.0)

        Z = (X - self.center_) / self.scale_

        return Z, np.isnan(Z)

    def _e_step(self, Z, mask, mu, sigma):
        """
        Fill missing entries with E[z_m | z_o] and accumulate the
        conditional covariance needed for the M-step.
        """

        d = Z.shape[1]
        Z_hat = np.where(mask, 0.0, Z)
        C = np.zeros((d, d))
        sigma_reg = sigma + self.ridge * np.eye(d)

        # rows with the same missingness pattern share the regression
        patterns, inverse = np.unique(mask, axis=0, return_inverse=True)
        inverse = inverse.ravel()

        for p, miss in enumerate(patterns):
            if not miss.any():
                continue

            rows = inverse == p
            obs = ~miss

            S_oo = sigma_reg[np.ix_(obs, obs)]
            S_mo = sigma_reg[np.ix_(miss, obs)]
            B = np.linalg.solve(S_oo, S_mo.T).T

            Z_hat[np.ix_(rows, miss)] = (
                mu[miss] + (Z[np.ix_(rows, obs)] - mu[obs]) @ B.T
            )
            C[np.ix_(miss, miss)] += rows.sum() * (
                sigma[np.ix_(miss, miss)] - B @ S_mo.T
            )

        return Z_hat, C

In [ ]:
num_cols = X_train.select_dtypes(include="float").columns # non-boolean features; target is not in X

imputer = EMImputer().fit(X_train[num_cols]) # fit on train only
X_train[num_cols] = imputer.transform(X_train[num_cols])
X_test[num_cols] = imputer.transform(X_test[num_cols])

print(list(num_cols), "EM iterations:", imputer.n_iter_)

## Best Random Forest parameters

These are the best parameters returned by the grid search you ran:

- `n_estimators=300`
- `min_samples_leaf=5`
- `max_features=1.0`
- `max_depth=None`
- `min_samples_split=2`
- `criterion="squared_error"`
- `ccp_alpha=0.0`

The recorded 5-fold CV results were RMSE ≈ 3883.76, MAE ≈ 2880.29, and R² ≈ 0.4408.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

best_params = {
    "n_estimators": 300,
    "criterion": "squared_error",
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 5,
    "max_features": 1.0,
    "ccp_alpha": 0.0,
    "bootstrap": True,
    "random_state": 1234,
    "n_jobs": 1,
}

best_model = RandomForestRegressor(**best_params)
best_model.fit(X_train, y_train)


## Test-set evaluation

In [ ]:
test_result = evaluate_model(
    best_model,
    X_test,
    y_test
)

test_result

## Optional: inspect feature importance

Feature importance is the Random Forest's impurity-based importance, not a causal effect.

In [ ]:
feature_importance = (
    pd.Series(best_model.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
)

feature_importance.head(20)